SEC Financial Dashboard Project

This project builds a financial performance dataset using SEC Financial Statement Data Sets from 2020 to 2025. The objective is to extract, clean, transform, and prepare real public company financial data for a Power BI dashboard.

The analysis focuses on five major U.S. companies:
Apple
Microsoft
Amazon
Tesla
JPMorgan Chase

In [142]:
import pandas as pd
import numpy as np
import os

In [144]:
base_path = os.path.expanduser("/Users/ankitatripathy/Desktop/PowerBi")

print("Folder exists:", os.path.exists(base_path))
print(sorted(os.listdir(base_path)))

Folder exists: True
['.DS_Store', '2020q1', '2020q2', '2020q3', '2020q4', '2021q1', '2021q2', '2021q3', '2021q4', '2022q1', '2022q2', '2022q3', '2022q4', '2023q1', '2023q2', '2023q3', '2023q4', '2024q1', '2024q2', '2024q3', '2024q4', '2025q1', '2025q2', '2025q3', '2025q4']


In [146]:
target_ciks = [
    320193,     # Apple
    789019,    # Microsoft
    1018724,   # Amazon
    1318605,   # Tesla
    19617      # JPMorgan Chase
]

financial_tags = [
    "Revenues",
    "SalesRevenueNet",
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "GrossProfit",
    "OperatingIncomeLoss",
    "NetIncomeLoss",
    "Assets",
    "Liabilities",
    "StockholdersEquity",
    "CashAndCashEquivalentsAtCarryingValue",
    "NetCashProvidedByUsedInOperatingActivities"
]

In [148]:
master_list = []

quarter_folders = sorted([
    folder for folder in os.listdir(base_path)
    if os.path.isdir(os.path.join(base_path, folder))
])

for folder in quarter_folders:
    print("Processing:", folder)

    sub_path = os.path.join(base_path, folder, "sub.txt")
    num_path = os.path.join(base_path, folder, "num.txt")

    if not os.path.exists(sub_path) or not os.path.exists(num_path):
        print("Missing files:", folder)
        continue

    sub = pd.read_csv(sub_path, sep="\t", low_memory=False)

    sub["cik"] = pd.to_numeric(sub["cik"], errors="coerce")

    sub_filtered = sub[
        (sub["cik"].isin(target_ciks)) &
        (sub["form"].isin(["10-K", "10-Q"]))
    ].copy()

    if sub_filtered.empty:
        continue

    needed_adsh = sub_filtered["adsh"].unique()

    num = pd.read_csv(num_path, sep="\t", low_memory=False)

    num_filtered = num[
        (num["adsh"].isin(needed_adsh)) &
        (num["tag"].isin(financial_tags)) &
        (num["uom"] == "USD") &
        (num["segments"].isna())
    ].copy()

    if num_filtered.empty:
        continue

    merged = pd.merge(
        num_filtered,
        sub_filtered[
            [
                "adsh",
                "name",
                "cik",
                "period",
                "filed",
                "form"
            ]
        ],
        on="adsh",
        how="inner"
    )

    merged["QuarterFolder"] = folder
    master_list.append(merged)

master = pd.concat(master_list, ignore_index=True)

print("Master shape:", master.shape)
master.head()

Processing: 2020q1
Processing: 2020q2
Processing: 2020q3
Processing: 2020q4
Processing: 2021q1
Processing: 2021q2
Processing: 2021q3
Processing: 2021q4
Processing: 2022q1
Processing: 2022q2
Processing: 2022q3
Processing: 2022q4
Processing: 2023q1
Processing: 2023q2
Processing: 2023q3
Processing: 2023q4
Processing: 2024q1
Processing: 2024q2
Processing: 2024q3
Processing: 2024q4
Processing: 2025q1
Processing: 2025q2
Processing: 2025q3
Processing: 2025q4
Master shape: (2922, 16)


,adsh,tag,version,ddate,qtrs,uom,segments,coreg,value,footnote,name,cik,period,filed,form,QuarterFolder
0,0001018724-20-000004,RevenueFromContractWithCustomerExcludingAssess...,us-gaap/2019,20180930,1,USD,NaN,NaN,5.657600e+10,NaN,AMAZON COM INC,1018724,20191231.0,20200131,10-K,2020q1
1,0001564590-20-002450,CashAndCashEquivalentsAtCarryingValue,us-gaap/2019,20190630,0,USD,NaN,NaN,1.135600e+10,NaN,MICROSOFT CORP,789019,20191231.0,20200129,10-Q,2020q1
2,0001564590-20-004475,NetIncomeLoss,us-gaap/2019,20171231,4,USD,NaN,NaN,-1.962000e+09,NaN,"TESLA, INC.",1318605,20191231.0,20200213,10-K,2020q1
3,0001564590-20-004475,CashAndCashEquivalentsAtCarryingValue,us-gaap/2019,20191231,0,USD,NaN,NaN,6.268000e+09,NaN,"TESLA, INC.",1318605,20191231.0,20200213,10-K,2020q1
4,0001018724-20-000004,RevenueFromContractWithCustomerExcludingAssess...,us-gaap/2019,20171231,4,USD,NaN,NaN,1.778660e+11,NaN,AMAZON COM INC,1018724,20191231.0,20200131,10-K,2020q1


In [150]:
master["ddate"] = pd.to_datetime(
    master["ddate"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

master["filed"] = pd.to_datetime(
    master["filed"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

master["Year"] = master["ddate"].dt.year
master["Quarter"] = master["ddate"].dt.quarter

master = master[
    (master["Year"] >= 2020) &
    (master["Year"] <= 2026)
].copy()

print("Master after date filter:", master.shape)
master.head()

Master after date filter: (2413, 18)


,adsh,tag,version,ddate,qtrs,uom,segments,coreg,value,footnote,name,cik,period,filed,form,QuarterFolder,Year,Quarter
161,0000019617-20-000299,NetCashProvidedByUsedInOperatingActivities,us-gaap/2019,2020-03-31,1,USD,NaN,NaN,-1.207720e+11,NaN,JPMORGAN CHASE & CO,19617,20200331.0,2020-05-07,10-Q,2020q2,2020,1
162,0001018724-20-000010,StockholdersEquity,us-gaap/2019,2020-03-31,0,USD,NaN,NaN,6.527200e+10,NaN,AMAZON COM INC,1018724,20200331.0,2020-05-01,10-Q,2020q2,2020,1
167,0001564590-20-019931,Liabilities,us-gaap/2019,2020-03-31,0,USD,NaN,NaN,2.651800e+10,NaN,"TESLA, INC.",1318605,20200331.0,2020-04-30,10-Q,2020q2,2020,1
169,0000019617-20-000299,Liabilities,us-gaap/2019,2020-03-31,0,USD,NaN,NaN,2.878169e+12,The following table presents information on as...,JPMORGAN CHASE & CO,19617,20200331.0,2020-05-07,10-Q,2020q2,2020,1
171,0001564590-20-019706,RevenueFromContractWithCustomerExcludingAssess...,us-gaap/2019,2020-03-31,3,USD,NaN,NaN,1.049820e+11,NaN,MICROSOFT CORP,789019,20200331.0,2020-04-29,10-Q,2020q2,2020,1


In [152]:
financial_table = master.pivot_table(
    index=[
        "name",
        "cik",
        "form",
        "filed",
        "ddate",
        "Year",
        "Quarter"
    ],
    columns="tag",
    values="value",
    aggfunc="first"
).reset_index()

financial_table.columns.name = None

print("Financial table shape:", financial_table.shape)
financial_table.head()

Financial table shape: (398, 17)


,name,cik,form,filed,ddate,Year,Quarter,Assets,CashAndCashEquivalentsAtCarryingValue,GrossProfit,Liabilities,NetCashProvidedByUsedInOperatingActivities,NetIncomeLoss,OperatingIncomeLoss,RevenueFromContractWithCustomerExcludingAssessedTax,Revenues,StockholdersEquity
0,AMAZON COM INC,1018724,10-K,2021-02-03,2020-03-31,2020,1,NaN,NaN,NaN,NaN,NaN,2.535000e+09,3.989000e+09,7.545200e+10,NaN,NaN
1,AMAZON COM INC,1018724,10-K,2021-02-03,2020-06-30,2020,2,NaN,NaN,NaN,NaN,NaN,5.243000e+09,5.843000e+09,8.891200e+10,NaN,NaN
2,AMAZON COM INC,1018724,10-K,2021-02-03,2020-09-30,2020,3,NaN,NaN,NaN,NaN,NaN,6.331000e+09,6.194000e+09,9.614500e+10,NaN,NaN
3,AMAZON COM INC,1018724,10-K,2021-02-03,2020-12-31,2020,4,3.211950e+11,4.212200e+10,NaN,NaN,6.606400e+10,7.222000e+09,2.289900e+10,1.255550e+11,NaN,9.340400e+10
4,AMAZON COM INC,1018724,10-K,2022-02-04,2020-12-31,2020,4,3.211950e+11,4.212200e+10,NaN,NaN,6.606400e+10,2.133100e+10,2.289900e+10,3.860640e+11,NaN,9.340400e+10


In [154]:
final_df = financial_table.copy()

final_df["Company"] = final_df["name"]

final_df["Revenue"] = np.nan

if "Revenues" in final_df.columns:
    final_df["Revenue"] = final_df["Revenue"].fillna(final_df["Revenues"])

if "SalesRevenueNet" in final_df.columns:
    final_df["Revenue"] = final_df["Revenue"].fillna(final_df["SalesRevenueNet"])

if "RevenueFromContractWithCustomerExcludingAssessedTax" in final_df.columns:
    final_df["Revenue"] = final_df["Revenue"].fillna(
        final_df["RevenueFromContractWithCustomerExcludingAssessedTax"]
    )

final_df["Gross_Profit"] = final_df.get("GrossProfit")
final_df["Operating_Income"] = final_df.get("OperatingIncomeLoss")
final_df["Net_Income"] = final_df.get("NetIncomeLoss")
final_df["Assets"] = final_df.get("Assets")
final_df["Liabilities"] = final_df.get("Liabilities")
final_df["Equity"] = final_df.get("StockholdersEquity")
final_df["Cash"] = final_df.get("CashAndCashEquivalentsAtCarryingValue")
final_df["Operating_Cash_Flow"] = final_df.get("NetCashProvidedByUsedInOperatingActivities")

In [156]:
company_map = {
    "APPLE INC": "Apple",
    "MICROSOFT CORP": "Microsoft",
    "AMAZON COM INC": "Amazon",
    "TESLA, INC.": "Tesla",
    "JPMORGAN CHASE & CO": "JPMorgan Chase"
}

final_df["Company"] = final_df["Company"].replace(company_map)

In [158]:
final_df["Profit_Margin"] = np.where(
    final_df["Revenue"] != 0,
    (final_df["Net_Income"] / final_df["Revenue"]) * 100,
    np.nan
)

final_df["Gross_Margin"] = np.where(
    final_df["Revenue"] != 0,
    (final_df["Gross_Profit"] / final_df["Revenue"]) * 100,
    np.nan
)

final_df["Operating_Margin"] = np.where(
    final_df["Revenue"] != 0,
    (final_df["Operating_Income"] / final_df["Revenue"]) * 100,
    np.nan
)

final_df["ROE"] = np.where(
    final_df["Equity"] != 0,
    (final_df["Net_Income"] / final_df["Equity"]) * 100,
    np.nan
)

final_df["Cash_Flow_Margin"] = np.where(
    final_df["Revenue"] != 0,
    (final_df["Operating_Cash_Flow"] / final_df["Revenue"]) * 100,
    np.nan
)

In [160]:
financial_performance_df = final_df[
    [
        "Company",
        "cik",
        "form",
        "filed",
        "ddate",
        "Year",
        "Quarter",
        "Revenue",
        "Gross_Profit",
        "Operating_Income",
        "Net_Income",
        "Operating_Cash_Flow",
        "Equity",
        "Profit_Margin",
        "Gross_Margin",
        "Operating_Margin",
        "ROE",
        "Cash_Flow_Margin"
    ]
].copy()

print("Full financial performance dataset:", financial_performance_df.shape)
financial_performance_df.head()

Full financial performance dataset: (398, 18)


,Company,cik,form,filed,ddate,Year,Quarter,Revenue,Gross_Profit,Operating_Income,Net_Income,Operating_Cash_Flow,Equity,Profit_Margin,Gross_Margin,Operating_Margin,ROE,Cash_Flow_Margin
0,Amazon,1018724,10-K,2021-02-03,2020-03-31,2020,1,7.545200e+10,NaN,3.989000e+09,2.535000e+09,NaN,NaN,3.359752,NaN,5.286805,NaN,NaN
1,Amazon,1018724,10-K,2021-02-03,2020-06-30,2020,2,8.891200e+10,NaN,5.843000e+09,5.243000e+09,NaN,NaN,5.896842,NaN,6.571666,NaN,NaN
2,Amazon,1018724,10-K,2021-02-03,2020-09-30,2020,3,9.614500e+10,NaN,6.194000e+09,6.331000e+09,NaN,NaN,6.584846,NaN,6.442353,NaN,NaN
3,Amazon,1018724,10-K,2021-02-03,2020-12-31,2020,4,1.255550e+11,NaN,2.289900e+10,7.222000e+09,6.606400e+10,9.340400e+10,5.752061,NaN,18.238222,7.732003,52.617578
4,Amazon,1018724,10-K,2022-02-04,2020-12-31,2020,4,3.860640e+11,NaN,2.289900e+10,2.133100e+10,6.606400e+10,9.340400e+10,5.525250,NaN,5.931400,22.837352,17.112189


In [162]:
annual_df = financial_performance_df[
    financial_performance_df["form"] == "10-K"
].copy()

annual_df = annual_df.sort_values(
    ["Company", "Year", "filed"]
)

annual_df = annual_df.drop_duplicates(
    subset=["Company", "Year"],
    keep="last"
)

print("Annual dataset shape:", annual_df.shape)

annual_df[
    [
        "Company",
        "Year",
        "Quarter",
        "Revenue",
        "Net_Income",
        "Operating_Cash_Flow",
        "Equity",
        "Profit_Margin",
        "ROE"
    ]
]

Annual dataset shape: (27, 18)


,Company,Year,Quarter,Revenue,Net_Income,Operating_Cash_Flow,Equity,Profit_Margin,ROE
9,Amazon,2020,4,NaN,NaN,NaN,9.340400e+10,NaN,NaN
13,Amazon,2021,4,NaN,NaN,NaN,1.382450e+11,NaN,NaN
14,Amazon,2022,4,5.139830e+11,-2.722000e+09,4.675200e+10,1.460430e+11,-0.529590,-1.863835
15,Amazon,2023,4,5.747850e+11,3.042500e+10,8.494600e+10,2.018750e+11,5.293284,15.071207
16,Amazon,2024,4,6.379590e+11,5.924800e+10,1.158770e+11,2.859700e+11,9.287117,20.718257
107,Apple,2020,3,NaN,NaN,NaN,6.533900e+10,NaN,NaN
111,Apple,2021,3,NaN,NaN,NaN,6.309000e+10,NaN,NaN
115,Apple,2022,3,NaN,NaN,NaN,5.067200e+10,NaN,NaN
116,Apple,2023,3,3.832850e+11,9.699500e+10,1.105430e+11,6.214600e+10,25.306234,156.076015
117,Apple,2024,3,3.910350e+11,9.373600e+10,1.182540e+11,5.695000e+10,23.971256,164.593503


In [164]:
print("Company count:")
print(annual_df["Company"].value_counts())

print("\nYear range:")
print(sorted(annual_df["Year"].unique()))

print("\nAvailable values:")
print(annual_df.notnull().sum().sort_values())

Company count:
Company
Apple             6
Microsoft         6
Amazon            5
JPMorgan Chase    5
Tesla             5
Name: count, dtype: int64

Year range:
[2020, 2021, 2022, 2023, 2024, 2025]

Available values:
Gross_Profit           14
Gross_Margin           14
Operating_Margin       17
Profit_Margin          17
Operating_Income       17
Revenue                17
Cash_Flow_Margin       17
ROE                    19
Net_Income             22
Operating_Cash_Flow    22
Equity                 24
Year                   27
ddate                  27
filed                  27
form                   27
cik                    27
Quarter                27
Company                27
dtype: int64


In [166]:
quarterly_df = financial_performance_df[
    financial_performance_df["form"] == "10-Q"
].copy()

quarterly_df = quarterly_df.sort_values(
    ["Company", "Year", "Quarter", "filed"]
)

quarterly_df = quarterly_df.drop_duplicates(
    subset=["Company", "Year", "Quarter"],
    keep="last"
)

quarterly_df.to_csv(
    "Financial_Performance_Dashboard_Quarterly_2020_2025.csv",
    index=False
)

print("Quarterly CSV exported successfully")
print(quarterly_df.shape)

Quarterly CSV exported successfully
(114, 18)


In [168]:
quarterly_df["Company"].value_counts()

Company
Amazon            23
JPMorgan Chase    23
Microsoft         23
Tesla             23
Apple             22
Name: count, dtype: int64

In [170]:
quarterly_df.notnull().sum().sort_values()

Gross_Profit            41
Gross_Margin            41
Operating_Margin        47
Profit_Margin           47
Operating_Income        47
Revenue                 47
Cash_Flow_Margin        47
ROE                     50
Net_Income              65
Operating_Cash_Flow     65
Equity                  95
Year                   114
ddate                  114
filed                  114
form                   114
cik                    114
Quarter                114
Company                114
dtype: int64

In [172]:
quarterly_df.head()

,Company,cik,form,filed,ddate,Year,Quarter,Revenue,Gross_Profit,Operating_Income,Net_Income,Operating_Cash_Flow,Equity,Profit_Margin,Gross_Margin,Operating_Margin,ROE,Cash_Flow_Margin
25,Amazon,1018724,10-Q,2021-07-30,2020-03-31,2020,1,NaN,NaN,NaN,NaN,NaN,6.527200e+10,NaN,NaN,NaN,NaN,NaN
30,Amazon,1018724,10-Q,2021-10-29,2020-06-30,2020,2,NaN,NaN,NaN,NaN,NaN,7.372800e+10,NaN,NaN,NaN,NaN,NaN
31,Amazon,1018724,10-Q,2021-10-29,2020-09-30,2020,3,2.605090e+11,NaN,6.194000e+09,6.331000e+09,5.529200e+10,8.277500e+10,2.430242,NaN,2.377653,7.648445,21.224603
45,Amazon,1018724,10-Q,2022-10-28,2020-12-31,2020,4,NaN,NaN,NaN,NaN,NaN,9.340400e+10,NaN,NaN,NaN,NaN,NaN
40,Amazon,1018724,10-Q,2022-07-29,2021-03-31,2021,1,NaN,NaN,NaN,NaN,NaN,1.033200e+11,NaN,NaN,NaN,NaN,NaN


In [174]:
powerbi_df = quarterly_df.copy()

powerbi_df["Sector"] = powerbi_df["Company"].map({
    "Amazon": "Technology",
    "Apple": "Technology",
    "Microsoft": "Technology",
    "Tesla": "Technology",
    "JPMorgan Chase": "Banking"
})

powerbi_df.shape

(114, 19)

In [176]:
powerbi_df.to_csv(
    "PowerBI_SEC_Dashboard_2020_2025.csv",
    index=False
)

print("Power BI file exported successfully")

Power BI file exported successfully


In [178]:
powerbi_df.head()

,Company,cik,form,filed,ddate,Year,Quarter,Revenue,Gross_Profit,Operating_Income,Net_Income,Operating_Cash_Flow,Equity,Profit_Margin,Gross_Margin,Operating_Margin,ROE,Cash_Flow_Margin,Sector
25,Amazon,1018724,10-Q,2021-07-30,2020-03-31,2020,1,NaN,NaN,NaN,NaN,NaN,6.527200e+10,NaN,NaN,NaN,NaN,NaN,Technology
30,Amazon,1018724,10-Q,2021-10-29,2020-06-30,2020,2,NaN,NaN,NaN,NaN,NaN,7.372800e+10,NaN,NaN,NaN,NaN,NaN,Technology
31,Amazon,1018724,10-Q,2021-10-29,2020-09-30,2020,3,2.605090e+11,NaN,6.194000e+09,6.331000e+09,5.529200e+10,8.277500e+10,2.430242,NaN,2.377653,7.648445,21.224603,Technology
45,Amazon,1018724,10-Q,2022-10-28,2020-12-31,2020,4,NaN,NaN,NaN,NaN,NaN,9.340400e+10,NaN,NaN,NaN,NaN,NaN,Technology
40,Amazon,1018724,10-Q,2022-07-29,2021-03-31,2021,1,NaN,NaN,NaN,NaN,NaN,1.033200e+11,NaN,NaN,NaN,NaN,NaN,Technology


In [180]:
import os

print(os.getcwd())

/Users/ankitatripathy


In [182]:
[f for f in os.listdir() if f.endswith(".csv")]

['india_tv_digital_arrest_articles.csv',
 'india_tv_digital_arrest_news.csv',
 'all_sub.csv',
 'PowerBI_SEC_Dashboard_2020_2025.csv',
 'indiatv_digital_arrest_news.csv',
 'toi_digital_arrest_articles.csv',
 'indiatv_digital_arrest_articles_google.csv',
 'temp_crosswalk_matched.csv',
 'indiatv_digital_arrest_scams.csv',
 'indiatv_digital_arrest_google.csv',
 'ndtv_digital_arrest_scraped.csv',
 'Financial_Performance_Dashboard_Quarterly_2020_2025.csv',
 'indiatv_digital_arrest_scraped.csv',
 'digital_arrest_selenium.csv',
 'i4c_digital_arrest_cases.csv',
 'final_crosswalk.csv',
 'indiatv_digital_arrest_articles.csv',
 'unmatched_plans.csv',
 'all_num.csv',
 'ndtv_digital_arrest_articles_google.csv']